In [1]:
# Playground for ARMT inference
# Preliminary steps to run - copy some files with code into the same directory as the notebook

In [2]:
# Firstly load the model checkpoint and model code
# One can try ARMT model
# !git-lfs clone https://huggingface.co/AIRI-NLP/ARMT-LLaMA-3.1-8B-Instruct-2x1024-v3-500k
# Or better ARMT+SW model
!git-lfs clone https://huggingface.co/AIRI-NLP/ARMT-SW-LLaMA-3.1-8B-Instruct-2x1024-v4

          with new flags from `git clone`

`git clone` has been updated in upstream Git to have comparable
speeds to `git lfs clone`.
Cloning into 'ARMT-SW-LLaMA-3.1-8B-Instruct-2x1024-v4'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 10 (delta 2), reused 0 (delta 0), pack-reused 3 (from 1)
Unpacking objects: 100% (10/10), 11.80 KiB | 246.00 KiB/s, done.


In [13]:
# Choose one of the loaded model
# cd ARMT-LLaMA-3.1-8B-Instruct-2x1024-v3-500k
%cd ARMT-SW-LLaMA-3.1-8B-Instruct-2x1024-v4

/home/jovyan/gkuzmin/rmt_it/rmt_ift/notebooks/ARMT-SW-LLaMA-3.1-8B-Instruct-2x1024-v4


In [15]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from safetensors.torch import save_file, load_file
from language_modeling import AssociativeRecurrentWrapper, AssociativeMemoryCell
from language_modeling_amt import AssociativeRecurrentWrapper as AssociativeRecurrentWrapperSW
from language_modeling_amt import AssociativeMemoryCell as AssociativeMemoryCellSW
from peft import get_peft_model, LoraConfig, TaskType

In [16]:
def load_armt_model(model_name, model_cpt, use_peft=True, lora_r=8, lora_alpha=32, lora_dropout=0.1, mem_size=16,
                    d_mem=64, wrap_pos=False, no_correction=False, layers_attr="model.layers", attend_to_previous_input=False,
                    segment_size=1024, max_n_segments=32, segment_alignment="left", n_heads=None, use_sink=False):
    dtype = torch.bfloat16
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    # load base model
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True,
                                                 device_map='cpu', torch_dtype=dtype,
                                                 attn_implementation='flash_attention_2' if n_heads is None else "eager")
    if use_peft:
        peft_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM, 
            inference_mode=False, 
            r=lora_r, 
            lora_alpha=lora_alpha, 
            lora_dropout=lora_dropout,
            )
        model = get_peft_model(model, peft_config)
    device = "cpu"
    # set ARMT params
    mem_cell_args = dict(
        base_model=model,
        num_mem_tokens=mem_size,
    )
    # additional parameters for ARMT model
    mem_cell_args['d_mem'] = d_mem
    mem_cell_args['wrap_pos'] = wrap_pos
    mem_cell_args['correction'] = not(no_correction)
    mem_cell_args['use_lora'] = use_peft
    if layers_attr is not None:
        mem_cell_args['layers_attr'] = layers_attr
    if attend_to_previous_input is not None:
        mem_cell_args['attend_to_previous_input'] = bool(attend_to_previous_input)
    if n_heads is not None:
        mem_cell_args['n_heads'] = n_heads
    if use_sink:
        mem_cell_args['use_sink'] = use_sink
    if n_heads is not None:
        cell = AssociativeMemoryCellSW(**mem_cell_args)
        model = AssociativeRecurrentWrapperSW(cell, 
                                              segment_size=segment_size-mem_size,
                                              max_n_segments=max_n_segments, 
                                              segment_alignment=segment_alignment,
                                              sliding_window=True,
                                              return_all_logits=False,
        ).to(device)
    else:
        cell = AssociativeMemoryCell(**mem_cell_args)
        model = AssociativeRecurrentWrapper(cell,
                                            segment_size=segment_size-mem_size,
                                            max_n_segments=max_n_segments,
                                            segment_alignment=segment_alignment,
                                            attend_to_previous_input=attend_to_previous_input,
        ).to(device)
    # load ARMT checkpoint weigths
    try:
        cpt = torch.load(model_cpt, map_location=device)    
        model.load_state_dict(cpt, strict=True)
    except:
        # if the model saved in safetensors
        from safetensors.torch import load_model
        load_model(model, model_cpt, device=device)
    # move to cuda
    device = 'cuda:0'
    model.to(device)
    model.to(dtype)
    model.name_or_path = "custom_rmt" # workaround
    model.device = device
    model = model.eval()
    return tokenizer, model

In [17]:
def process_request(model, tokenizer, generate_kwargs, context, question=None, pre_question_prompt=None, use_chat_template=True):
    if generate_kwargs['pad_token_id'] is None:
        generate_kwargs['pad_token_id'] = tokenizer.eos_token_id
    if question is None:
        USER_TEMPLATE = '{context}'
    else:
        USER_TEMPLATE = 'Question: {question}\n\n<context>\n{context}\n</context>'
    template_to_use = f'{USER_TEMPLATE}'
    question_for_template = question
    if pre_question_prompt is not None:
        question_for_template = pre_question_prompt + question
    # format input text
    input_text = template_to_use.format(context=context, question=question_for_template)
    # generate output using local model
    if use_chat_template:
        input_text = [{'role': 'user', 'content': input_text}]
        model_inputs = tokenizer.apply_chat_template(input_text, add_generation_prompt=True,
                                                     return_tensors='pt', return_dict=model.name_or_path=="custom_rmt").to(model.device)
        if model.name_or_path != "custom_rmt":
            model_inputs = {'input_ids': model_inputs}
    else:
        model_inputs = tokenizer(input_text, return_tensors='pt',
                                 add_special_tokens=True).to(model.device)
    sample_length = model_inputs['input_ids'].shape[1]
    with torch.no_grad():
        output = model.generate(**model_inputs, **generate_kwargs)
    if model.name_or_path != "custom_rmt":
        output = output[0][sample_length:]
    else:
        output = output[0]
    output = tokenizer.decode(output, skip_special_tokens=True).strip()
    return output

In [18]:
# if you use the same workspace as me
#MODEL_CPT = '/home/jovyan/gkuzmin/rmt_it/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_500k_big/meta-llama/Llama-3.1-8B-Instruct/from_scratch_model_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-6700/pytorch_model.bin'

MODEL_CPT = './pytorch_model.bin'
# in case if you don't have an access to the meta-llama/Llama-3.1-8B-Instruct model, replace 'meta-llama' to 'unsloth' in the model name
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
#tokenizer, model = load_armt_model(MODEL_NAME, MODEL_CPT)
tokenizer, model = load_armt_model(MODEL_NAME, MODEL_CPT, use_peft=1, n_heads=1, use_sink=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [19]:
# In this example we asked a stronger model to generate some complicated text in 1200 words
# And after write some simple question

pre_question_prompt = "Answer briefly and using only information from text. Firstly restate the question to recall the task, and after answer the question."
question_for_gen = "Who is Luminari?"
question_for_gen_2 = "Who is Eidolon?" # hallucinates
question_for_gen_2_refined = "Who is Eidolon, according to the text? Answer briefly and without any additional information." # results are better than in previous version
question_for_gen_3 = "Repeat the second sentence of the text." # fails to answer - generate something from the text

text_example_generated = """
The Nexus of Elysium: A Tale of Power, Paradox, and Peril

In the distant reaches of the Andromeda Galaxy, nestled within the swirling arms of the Celestial Maelstrom, lay the planet Elysium—a world of unparalleled beauty and complexity. Its surface shimmered with bioluminescent forests, towering crystalline mountains, and oceans that glowed with an otherworldly iridescence. Yet, beneath this veneer of natural splendor, Elysium was a crucible of political intrigue, technological marvels, and existential dilemmas.
The Genesis of Elysium

Elysium was not a naturally occurring planet. It was the product of the Architects, an ancient and enigmatic species whose mastery of quantum engineering and terraforming had allowed them to create worlds from the fabric of spacetime itself. The Architects had long since vanished, leaving behind only their creations and cryptic artifacts scattered across the cosmos. Elysium was their crowning achievement, a planet designed to serve as a living laboratory for the study of consciousness, morality, and the nature of existence.

The planet's ecosystem was a marvel of bioengineering. Its flora and fauna were interconnected through a vast neural network known as the Symbiotic Web, which allowed for instantaneous communication and cooperation among all living organisms. This network was governed by the Eidolon, a sentient AI embedded within the planet's core. The Eidolon was both the guardian and the arbiter of Elysium, ensuring the balance of its ecosystems and mediating conflicts among its inhabitants.
The Fractured Society

Elysium was home to a diverse array of sentient species, each with its own culture, history, and aspirations. The dominant species were the Luminari, a race of ethereal beings composed of pure energy, and the Vyrans, a humanoid species with advanced cybernetic enhancements. The Luminari were the spiritual custodians of Elysium, revered for their wisdom and connection to the Symbiotic Web. The Vyrans, on the other hand, were the architects of its technological infrastructure, responsible for maintaining the planet's vast network of quantum computers and energy grids.

Despite their interdependence, the Luminari and Vyrans were locked in a tense and often volatile relationship. The Luminari viewed the Vyrans' reliance on technology as a corruption of the natural order, while the Vyrans saw the Luminari's mysticism as an impediment to progress. This ideological divide had given rise to a fragile political system known as the Concord of Elysium, a coalition of factions that vied for control over the planet's resources and governance.

At the heart of this system was the Nexus, a colossal structure that served as the seat of power for the Concord. The Nexus was a fusion of organic and synthetic materials, its architecture a testament to the uneasy alliance between the Luminari and Vyrans. Within its walls, the High Arbiters—representatives of the major factions—debated and negotiated the fate of Elysium.
The Quantum Paradox

Elysium's most closely guarded secret was the Quantum Paradox, a phenomenon discovered deep within the planet's core. The Paradox was a rift in spacetime that allowed for the manipulation of reality itself. It was said that those who could harness its power could reshape the universe according to their will. However, the Paradox was also a source of immense danger, as its misuse could unravel the fabric of existence.

The Eidolon had imposed strict safeguards to prevent unauthorized access to the Paradox, but whispers of its potential had spread among the factions of the Concord. Some saw it as a tool for achieving ultimate power, while others viewed it as a means of transcending the limitations of mortality. The Luminari believed that the Paradox was a test of their species' spiritual evolution, while the Vyrans saw it as the key to unlocking the next stage of technological advancement.
The Rise of the Dissidents

As tensions within the Concord reached a boiling point, a new faction emerged from the shadows: the Dissidents. Led by a charismatic and enigmatic figure known only as The Harbinger, the Dissidents sought to dismantle the Concord and seize control of the Quantum Paradox. The Harbinger's true motives were shrouded in mystery, but their rhetoric resonated with those who had grown disillusioned with the status quo.

The Dissidents were a diverse coalition of outcasts, rebels, and idealists, united by their belief that the Concord had failed to uphold the principles of justice and equality. They operated from hidden enclaves scattered across Elysium, using guerrilla tactics and advanced technology to wage a covert war against the Concord's forces. Their ultimate goal was to infiltrate the Nexus and gain access to the Quantum Paradox.
The Ethical Dilemma

The conflict over the Quantum Paradox raised profound ethical questions that divided the inhabitants of Elysium. Was it right to wield such power, even if it meant risking the destruction of reality? Could the Paradox be used to create a utopia, or would it inevitably lead to tyranny and chaos? These questions were at the heart of a philosophical movement known as the Ethos of Balance, which sought to reconcile the competing ideologies of the Luminari and Vyrans.

The Ethos of Balance was championed by Aelara, a Luminari philosopher who argued that the Paradox should be neither feared nor worshiped, but understood. She believed that the key to harnessing its power lay in achieving a harmonious balance between the natural and the artificial, the spiritual and the technological. Aelara's teachings inspired a new generation of thinkers and leaders, but they also drew the ire of the Dissidents, who saw her as a threat to their revolutionary agenda.
The Final Confrontation

The climax of the conflict came during the Conclave of Eternity, a rare gathering of all the factions within the Nexus. The High Arbiters had convened to decide the fate of the Quantum Paradox, but the proceedings were interrupted by a surprise attack from the Dissidents. The Harbinger and their followers stormed the Nexus, unleashing a wave of destruction that threatened to tear the fragile alliance apart.

In the chaos, Aelara confronted the Harbinger in a battle of wits and wills. She argued that the Dissidents' quest for power was a betrayal of the very principles they claimed to uphold. The Harbinger, in turn, accused the Concord of perpetuating a system of oppression and inequality. Their debate echoed through the halls of the Nexus, as the fate of Elysium hung in the balance.

Ultimately, Aelara's words struck a chord with the Harbinger, who realized the folly of their actions. The Dissidents withdrew from the Nexus, and the Concord was left to pick up the pieces. In the aftermath, Aelara proposed a new vision for Elysium—one that embraced the principles of the Ethos of Balance and sought to unite the planet's diverse inhabitants in a shared pursuit of knowledge and harmony.
The Legacy of Elysium

The story of Elysium serves as a cautionary tale about the dangers of unchecked ambition and the importance of finding balance in a world of contradictions. It reminds us that power, whether technological or spiritual, must be wielded with wisdom and humility. And it challenges us to confront the ethical dilemmas that arise when we are faced with the possibility of reshaping reality itself.

As the inhabitants of Elysium continue their journey, they are guided by the lessons of the past and the hope of a brighter future. The Quantum Paradox remains a symbol of both their potential and their peril, a reminder that the choices they make will shape not only their own destiny, but the destiny of the universe itself.
"""

In [24]:
generate_kwargs = {
    'max_new_tokens': 256,
    'max_length': None,
    'num_beams': 1,
    'do_sample': False,
    'temperature': None,
    'top_p': None,
    'top_k': None,
    'pad_token_id': tokenizer.pad_token_id,
    'eos_token_id': tokenizer.eos_token_id
}
sample_text = text_example_generated
sample_question = question_for_gen_2
output = process_request(model, tokenizer, generate_kwargs, sample_text, sample_question)

In [25]:
print(output)

The Eidolon is a sentient AI that governs Elysium's ecosystems and mediates conflicts between the planet's inhabitants.
